In [6]:
API_KEY ='d30p739r01qnu2qv1tmgd30p739r01qnu2qv1tn0'

In [7]:
#### 뉴스 데이터 수집

import os
import time
import requests
import csv
from datetime import datetime, timedelta
from collections import Counter

API_KEY = os.getenv('FINN_API_KEY')
BASE_URL = 'https://finnhub.io/api/v1/company-news'

def safe_convert_timestamp(timestamp):
    try:
        if timestamp is None or not isinstance(timestamp, (int, float)) or timestamp <= 0:
            return ''
        if timestamp > 32503680000:  # 3000-01-01 00:00:00 UTC 제한
            return ''
        return datetime.fromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ''

def collect_news_data(symbol, from_date, to_date, max_calls_per_minute=60):
    all_news = []
    call_count = 0
    current_from = datetime.strptime(from_date, "%Y-%m-%d")
    current_to = datetime.strptime(to_date, "%Y-%m-%d")
    
    while current_from <= current_to:
        from_str = current_from.strftime("%Y-%m-%d")
        to_str = (current_from + timedelta(days=7)).strftime("%Y-%m-%d")
        if current_from + timedelta(days=7) > current_to:
            to_str = to_date
        
        params = {
            'symbol': symbol,
            'from': from_str,
            'to': to_str,
            'token': API_KEY
        }
        
        response = requests.get(BASE_URL, params=params)
        call_count += 1
        
        if response.status_code == 200:
            news_list = response.json()
            for news in news_list:
                timestamp = news.get('datetime')
                readable_date = safe_convert_timestamp(timestamp)
                
                data = {
                    'date': readable_date,
                    'title': news.get('headline', ''),
                    'summary': news.get('summary', ''),
                    'related': news.get('related', symbol)
                }
                all_news.append(data)
        else:
            print(f"API 호출 오류 {response.status_code} - {response.text}")
        
        if call_count >= max_calls_per_minute:
            print("60회 호출 도달, 60초 대기 중...")
            time.sleep(60)
            call_count = 0
        
        current_from += timedelta(days=7)

    return all_news

def save_news_to_csv(news_data, filename):
    with open(filename, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['date', 'title', 'summary', 'related'])
        writer.writeheader()
        for record in news_data:
            writer.writerow(record)
    print(f"{filename} 파일에 저장 완료")

def count_news_by_company(news_data):
    symbols = [news['related'] for news in news_data if 'related' in news]
    counts = Counter(symbols)
    return counts

# 전체 뉴스 데이터를 모아서 한 번에 저장하며, 기업별 건수 출력
symbols = ['NVDA', 'MSFT', 'AAPL']
from_date = '2020-01-01'
to_date = '2024-12-31'

all_news_data = []

for symbol in symbols:
    print(f"{symbol} 뉴스 수집 시작...")
    news_data = collect_news_data(symbol, from_date, to_date)
    all_news_data.extend(news_data)
    counts = count_news_by_company(news_data)
    print(f"{symbol} 뉴스 건수: {counts.get(symbol, 0)}개")

save_news_to_csv(all_news_data, "news_data.csv")

NVDA 뉴스 수집 시작...
API 호출 오류 401 - {"error":"Please use an API key."}
API 호출 오류 401 - {"error":"Please use an API key."}
API 호출 오류 401 - {"error":"Please use an API key."}
API 호출 오류 401 - {"error":"Please use an API key."}


KeyboardInterrupt: 

In [12]:
import yfinance as yf
import pandas as pd

symbols = ['NVDA', 'MSFT', 'AAPL']
from_date = '2020-01-01'
to_date = '2024-12-31'

all_stock_data = []

for symbol in symbols:
    print(f"{symbol} 주가 데이터 수집 중...")
    df = yf.download(symbol, start=from_date, end=to_date)

    # MultiIndex 컬럼 평탄화
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]  # 두 번째 레벨 무시

    df = df.reset_index()
    df['Symbol'] = symbol

    # 필요한 컬럼만 단일화
    df = df[['Symbol', 'Date', 'Open', 'Close']]
    all_stock_data.append(df)

result = pd.concat(all_stock_data, ignore_index=True)

print("Concat 결과 컬럼:", result.columns.tolist())

result.to_csv("stock_data.csv", index=False, encoding='utf-8')
print("stock_data.csv 파일 저장 완료")


C:\Users\jinfo\AppData\Local\Temp\ipykernel_15116\2085453405.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=from_date, end=to_date)
[*********************100%***********************]  1 of 1 completed
C:\Users\jinfo\AppData\Local\Temp\ipykernel_15116\2085453405.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=from_date, end=to_date)
[*********************100%***********************]  1 of 1 completed
C:\Users\jinfo\AppData\Local\Temp\ipykernel_15116\2085453405.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, start=from_date, end=to_date)
[*********************100%***********************]  1 of 1 completed

NVDA 주가 데이터 수집 중...
MSFT 주가 데이터 수집 중...
AAPL 주가 데이터 수집 중...
Concat 결과 컬럼: ['Symbol', 'Date', 'Open', 'Close']


stock_data.csv 파일 저장 완료


In [2]:
pip install sklearn

  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://g

In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np
from sklearn.model_selection import train_test_split

# 1. 뉴스 데이터 로드 및 텍스트 전처리 / 임베딩
news = pd.read_csv('news_data.csv')
news['date'] = pd.to_datetime(news['date']).dt.date
news['text'] = news['title'].fillna('') + ' ' + news['summary'].fillna('')
news['related'] = news['related'].fillna('')

# 2. 주가 데이터 로드 및 익일 수익률 계산
stock = pd.read_csv('stock_data.csv')
stock['Date'] = pd.to_datetime(stock['Date']).dt.date
stock = stock.sort_values(by=['Symbol', 'Date'])
stock['next_close'] = stock.groupby('Symbol')['Close'].shift(-1)
stock['log_return'] = np.log(stock['next_close'] / stock['Close'])
stock = stock.dropna(subset=['log_return'])

# 3. 뉴스와 주가 매칭
data = pd.merge(news, stock, how='inner', left_on=['date', 'related'], right_on=['Date', 'Symbol'])

# 4. FINBERT 임베딩 함수 정의
tokenizer = AutoTokenizer.from_pretrained('yiyanghkust/finbert-tone')
model = AutoModel.from_pretrained('yiyanghkust/finbert-tone')

def get_finbert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] 토큰 임베딩 추출
    cls_embed = outputs.last_hidden_state[:, 0, :]
    return cls_embed.squeeze().numpy()


FileNotFoundError: [Errno 2] No such file or directory: 'news_data.csv'

In [ ]:
# 5. 뉴스 텍스트 임베딩 생성 (주의: 느릴 수 있어 배치 처리 권장)
data['embedding'] = data['text'].map(get_finbert_embedding)

In [ ]:
# 6. PyTorch Dataset 정의
class StockNewsDataset(Dataset):
    def __init__(self, df):
        self.embeddings = list(df['embedding'])
        self.targets = list(df['log_return'])
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, idx):
        return torch.tensor(self.embeddings[idx], dtype=torch.float32), torch.tensor(self.targets[idx], dtype=torch.float32)

# 7. 데이터 분할 및 DataLoader 생성
train_df, val_df = train_test_split(data, test_size=0.2, random_state=42)
train_dataset = StockNewsDataset(train_df)
val_dataset = StockNewsDataset(val_df)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# 8. 단순한 회귀용 MLP 모델 정의
class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLPRegressor(input_dim=768).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# 9. 학습 반복문(간략)
for epoch in range(10):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Train Loss: {total_loss/len(train_loader)}")

# 10. 이후 평가, 하이퍼파라미터 최적화, 분류 등 확장 가능
